## 02 — Biomarkers + baseline (logistic or random forest)

Runs **`make index`** (refreshes **`results/dataset_index.csv`** with paths + placeholder **`label`**), **`make biomarkers`**, and **`make train`** using the **`scd-octa-screening`** repo root (detected via `Makefile` / `src/scd_octa`, so it works whether your kernel cwd is **`notebooks/`**, the **repo root**, or the parent **`Hospital/`** folder).

If **`../OCTA AI`** exists (flat TIFF folder next to this repo), the same path is used for index and biomarkers so row counts stay aligned. Otherwise the Makefile default **`data/raw/scd-data`** is used (symlink your cohort there, or pass `DATA_ROOT` yourself).

**Labels:** `make index` writes a deterministic placeholder **`label`** (0/1 per `subject_id`) so baseline training can run; replace with real screening outcomes when available.

Outputs:
- `results/biomarkers_vessel_density.csv`
- `results/baseline/train_eval_report.json`
- `results/baseline/test_predictions.csv` (when enough labeled subjects)
- `models/baseline_logistic.joblib` (default) or `models/baseline_random_forest.joblib` if you run `make train MODEL=random_forest`

In [2]:
import subprocess
from pathlib import Path


def find_scd_octa_repo() -> Path:
    """Resolve repo root whether cwd is `notebooks/`, repo root, or parent `Hospital/`."""
    cwd = Path.cwd().resolve()
    anchors = [cwd]
    nested = cwd / "scd-octa-screening"
    if nested.is_dir():
        anchors.append(nested.resolve())

    seen: set = set()
    for start in anchors:
        chain = [start, *start.parents]
        for d in chain:
            if d in seen:
                continue
            seen.add(d)
            if (d / "Makefile").is_file() and (d / "src" / "scd_octa").is_dir():
                return d
    raise RuntimeError(
        "Could not find scd-octa-screening (no Makefile + src/scd_octa). "
        "cd into scd-octa-screening or open the notebook from that repo."
    )


ROOT = find_scd_octa_repo()
# Flat TIFF folder: sibling of repo under the same parent (e.g. Hospital/OCTA AI)
octa_ai = ROOT.parent / "OCTA AI"
if octa_ai.is_dir():
    dr = str(octa_ai.resolve())
    print("DATA_ROOT override:", dr)
    data_root_arg = f"DATA_ROOT={dr}"
else:
    print("No OCTA AI next to repo — Makefile default ./data/raw/scd-data")
    data_root_arg = None

print("Repo ROOT:", ROOT)
print("Kernel cwd:", Path.cwd())

if data_root_arg:
    subprocess.run(["make", "index", data_root_arg], check=True, cwd=ROOT)
    subprocess.run(["make", "biomarkers", data_root_arg], check=True, cwd=ROOT)
    subprocess.run(["make", "train", data_root_arg], check=True, cwd=ROOT)
else:
    subprocess.run(["make", "index"], check=True, cwd=ROOT)
    subprocess.run(["make", "biomarkers"], check=True, cwd=ROOT)
    subprocess.run(["make", "train"], check=True, cwd=ROOT)

DATA_ROOT override: /Users/tripa/Desktop/Projects/Hospital/OCTA AI
Repo ROOT: /Users/tripa/Desktop/Projects/Hospital/scd-octa-screening
Kernel cwd: /Users/tripa/Desktop/Projects/Hospital/scd-octa-screening/notebooks
PYTHONPATH=./src /Users/tripa/Desktop/Projects/Hospital/scd-octa-screening/.venv/bin/python -m scd_octa.make_index --data-root "/Users/tripa/Desktop/Projects/Hospital/OCTA AI" --out-dir "./results"
Wrote: results/dataset_index.csv
Wrote: results/dataset_summary.json
PYTHONPATH=./src /Users/tripa/Desktop/Projects/Hospital/scd-octa-screening/.venv/bin/python -m scd_octa.compute_biomarkers --data-root "/Users/tripa/Desktop/Projects/Hospital/OCTA AI" --out-dir "./results"
Wrote: results/biomarkers_vessel_density.csv
PYTHONPATH=./src /Users/tripa/Desktop/Projects/Hospital/scd-octa-screening/.venv/bin/python -m scd_octa.train_baseline \
		--biomarkers-csv "./results/biomarkers_vessel_density.csv" \
		--labels-csv "./results/dataset_index.csv" \
		--out-dir "./results/baseline" \
